# 05 - Silver Pipeline (Lakeflow Declarative Pipeline)

Cleans, standardizes, and de-duplicates Bronze into Silver. Also builds the
two SCD Type 2 dimensions (stores, customers).

This is a **pipeline library notebook** — it does not run on its own. It
only runs inside the Lakeflow pipeline declared in `resources/pipelines.yml`,
which supplies `dlt` and `spark` automatically. The `@dlt.table` /
`@dlt.expect_*` decorators here are the Databricks platform's own syntax
for Lakeflow pipelines, not something we added — every Lakeflow pipeline
is written this way.

Quality rules are loaded straight from `config/quality_rules.yaml` below —
one plain function, no extra framework.

In [ ]:
import dlt
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = spark.conf.get("catalog_name", "maven_test")

### Load the quality rule catalog

Every rule lives in `config/quality_rules.yaml` (one file, so Silver's
expectations and the human-readable DQ report always agree). This just
reads the file and returns the rules that apply to one table and one
action (`warn` / `quarantine` / `fail`), as a plain dict for `@dlt.expect_all*`.

In [ ]:
with open("config/quality_rules.yaml") as f:
    rule_catalog = yaml.safe_load(f)["rules"]


def rules_for(table, action):
    out = {}
    for r in rule_catalog:
        if r["table"] == table and r["action"] == action:
            out[r["rule_id"]] = r["predicate_sql"]
    return out

### Reference / dimension sources

Light cleansing only — type casts and trimming whitespace.

In [ ]:
@dlt.table(name="regions_clean", comment="Cleansed regions reference data.")
def regions_clean():
    return dlt.read(f"{CATALOG}.bronze.adls_regions").select(
        F.col("region_id").cast("int"),
        F.trim("sales_district").alias("sales_district"),
        F.trim("sales_region").alias("sales_region"),
    )


@dlt.table(name="calendar_clean", comment="Calendar dimension with derived date parts.")
def calendar_clean():
    df = dlt.read(f"{CATALOG}.bronze.adls_calendar").withColumn("date", F.to_date("date"))
    return (
        df.withColumn("year", F.year("date"))
        .withColumn("quarter", F.quarter("date"))
        .withColumn("month", F.month("date"))
        .withColumn("day_of_week", F.dayofweek("date"))
        .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
    )


@dlt.table(name="promotions_clean", comment="Cleansed promotions.")
@dlt.expect_or_drop("PROMO_VALID_WINDOW", "end_date >= start_date")
def promotions_clean():
    return dlt.read(f"{CATALOG}.bronze.adls_promotions").select(
        "promotion_id", "promotion_name",
        F.to_date("start_date").alias("start_date"),
        F.to_date("end_date").alias("end_date"),
        F.col("discount_pct").cast("decimal(5,2)"),
        "channel",
    )


@dlt.table(name="products_clean", comment="Cleansed product master (full-load source, no SCD2).")
def products_clean():
    return dlt.read(f"{CATALOG}.bronze.mongo_product_master").select(
        F.col("product_id").cast("int"),
        "product_brand", "product_name", "product_sku", "product_category",
        F.col("product_retail_price").cast("decimal(10,2)"),
        F.col("product_cost").cast("decimal(10,2)"),
    )

### Stores — SCD Type 2

Tracks store history (store type/size/name changes) over time. Business
key `store_id`, sequenced by `last_remodel_date` (falls back to
`first_opened_date` for the opening record). Dirty country-name variants
("usa", "U.S.A.") are normalized here rather than dropped.

In [ ]:
@dlt.view(name="stores_scd2_source")
def stores_scd2_source():
    df = dlt.read(f"{CATALOG}.bronze.adls_stores")
    country_map = {
        "usa": "USA", "usa ": "USA", "united states": "USA", "u.s.a.": "USA", "us": "USA",
        "mex": "Mexico", "mexico ": "Mexico", "mx": "Mexico",
        "canada ": "Canada", "can": "Canada",
    }
    mapping_expr = F.create_map([F.lit(x) for pair in country_map.items() for x in pair])
    normalized_country = F.coalesce(
        mapping_expr[F.lower(F.trim(F.col("store_country")))],
        F.col("store_country"),
    )
    return df.withColumn("store_country", normalized_country).withColumn(
        "effective_sequence",
        F.coalesce(F.to_date("last_remodel_date"), F.to_date("first_opened_date")),
    )


dlt.create_streaming_table("stores_scd2")
dlt.apply_changes(
    target="stores_scd2",
    source="stores_scd2_source",
    keys=["store_id"],
    sequence_by="effective_sequence",
    stored_as_scd_type=2,
)

### Customers — SCD Type 2

Business key `customer_id`, sequenced by the generator's
`_source_last_modified` field.

In [ ]:
@dlt.view(name="customers_scd2_source")
def customers_scd2_source():
    return dlt.read(f"{CATALOG}.bronze.mongo_customer_master").withColumn(
        "effective_sequence", F.to_timestamp("_source_last_modified")
    )


dlt.create_streaming_table("customers_scd2")
dlt.apply_changes(
    target="customers_scd2",
    source="customers_scd2_source",
    keys=["customer_id"],
    sequence_by="effective_sequence",
    stored_as_scd_type=2,
)

### Transactions

De-duplicated on `transaction_id` (keeping the latest ingestion), with
the quality rules from `config/quality_rules.yaml` applied as Lakeflow
expectations — `warn` rows are kept and flagged, `quarantine` rows are
dropped (and separately written to `ops.quarantine`), `fail` rows stop
the whole pipeline update.

In [ ]:
@dlt.table(name="transactions_clean", comment="Cleansed, deduplicated transactions with quality rules applied.")
@dlt.expect_all(rules_for("silver.transactions_clean", "warn"))
@dlt.expect_all_or_drop(rules_for("silver.transactions_clean", "quarantine"))
@dlt.expect_all_or_fail(rules_for("silver.transactions_clean", "fail"))
def transactions_clean():
    df = (
        dlt.read(f"{CATALOG}.bronze.adls_transactions")
        .withColumn("transaction_date", F.to_date("transaction_date"))
        .withColumn("stock_date", F.to_date("stock_date"))
        .withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("discount_amount", F.col("discount_amount").cast("decimal(10,2)"))
    )
    rank = F.row_number().over(
        Window.partitionBy("transaction_id").orderBy(F.col("ingestion_timestamp").desc())
    )
    return df.withColumn("_dedup_rank", rank).filter("_dedup_rank = 1").drop("_dedup_rank")


@dlt.table(name="returns_clean", comment="Cleansed returns with validity checks.")
@dlt.expect_all_or_drop(rules_for("silver.returns_clean", "quarantine"))
def returns_clean():
    return (
        dlt.read(f"{CATALOG}.bronze.adls_returns")
        .withColumn("return_date", F.to_date("return_date"))
        .withColumn("quantity", F.col("quantity").cast("int"))
    )

### Streaming facts from Kafka

De-dup and watermarking already happened at the Bronze Kafka consumer —
Silver only applies the business-rule expectations here, so we don't do
the same de-dup twice.

In [ ]:
@dlt.table(name="online_orders_clean", comment="Cleansed online orders stream.")
@dlt.expect_all_or_drop(rules_for("silver.online_orders_clean", "quarantine"))
def online_orders_clean():
    return dlt.read_stream(f"{CATALOG}.bronze.kafka_online_orders_raw").withColumn(
        "quantity", F.col("quantity").cast("int")
    )


@dlt.table(name="inventory_restock_clean", comment="Cleansed inventory restock stream.")
def inventory_restock_clean():
    return dlt.read_stream(f"{CATALOG}.bronze.kafka_inventory_restock_raw").withColumn(
        "restock_quantity", F.col("restock_quantity").cast("int")
    )